# Trees

*Run a cell with ▶ (or Shift+Enter): Sphinx runs the Java with your JDK. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Represent a tree with a parent array or with nodes, and find its levels
- Build a binary search tree and traverse it in-order, pre-order and post-order
- Explain why a BST's height decides its speed, and when it turns into a list
- Answer prefix questions with a trie, or with sorted words
- Add values up a tree with a post-order traversal

## In short

A **tree** is a hierarchy: one **root**, and every other node has exactly one **parent**. Org charts, folders, web pages and the expressions a compiler reads are trees. A node's **depth** is its distance from the root, and the tree's **height** is the depth of its deepest node. A tree can live in a **parent array** (`parent[i]`) or in **nodes** that point to their children.

**A tree in a parent array:** each node's depth comes from its parent's, and its children are counted in one pass.

In [ ]:
int[] parent = {-1, 0, 0, 1, 1, 2, 4, 4}; // node 0 is the root
int[] depth = new int[parent.length];
int[] children = new int[parent.length];
for (int i = 1; i < parent.length; i++) {
    depth[i] = depth[parent[i]] + 1;
    children[parent[i]]++;
}
for (int i = 0; i < parent.length; i++) {
    IO.println("node " + i + ": depth " + depth[i] + ", children " + children[i] + (children[i] == 0 ? " (a leaf)" : ""));
}

A **binary search tree** (BST) keeps smaller keys on the left of every node and bigger keys on the right, so a search walks down a single path: O(height).

![A binary search tree and its in-order traversal](bst.svg)

Random insertions give a height of about 2 log₂ n. Sorted insertions build a chain of height n: a slow linked list. That's why Java's `TreeMap` rebalances itself to stay O(log n).

**Random against sorted insertions.** This cell only tracks how deep each new key lands; BST Builder asks for the whole tree.

In [ ]:
int n = 2000;
for (String order : new String[] {"random", "sorted"}) {
    int[] keys = order.equals("random") ? new Random(5).ints(n, 0, 1_000_000).distinct().toArray() : java.util.stream.IntStream.range(0, n).toArray();
    TreeMap<Integer, Integer> depthOf = new TreeMap<>(); // key -> its depth in the BST
    int height = 0;
    for (int key : keys) {
        // In a BST a new key hangs below its closest neighbours: the deeper of the two.
        Map.Entry<Integer, Integer> below = depthOf.lowerEntry(key);
        Map.Entry<Integer, Integer> above = depthOf.higherEntry(key);
        int depth = 1 + Math.max(below == null ? 0 : below.getValue(), above == null ? 0 : above.getValue());
        depthOf.put(key, depth);
        height = Math.max(height, depth);
    }
    IO.println(String.format("%s insertions of %d keys: height %d (log2 n is about %.0f)", order, keys.length, height, Math.log(keys.length) / Math.log(2)));
}

Three **traversals** visit every node:

- **pre-order** (node, left, right) copies a tree;
- **in-order** (left, node, right) gives a BST's keys sorted;
- **post-order** (left, right, node) finishes the children first, which is how a folder adds up its contents.

A **trie** is a tree of letters: each path from the root spells a prefix, so counting the words that start with "car" takes three steps down.

**Words by prefix in a sorted set:** the words starting with "ca" sit between "ca" and "ca{" (`{` comes right after `z`).

In [ ]:
TreeSet<String> words = new TreeSet<>(List.of("car", "card", "care", "cart", "cat", "dog", "do", "dot"));
for (String prefix : new String[] {"ca", "car", "do", "x"}) {
    IO.println(prefix + " -> " + words.subSet(prefix, prefix + "{"));
}

Watch out: recursion on a deep, chain-shaped tree can overflow the call stack.

**Deep recursion:** walking a chain of 100,000 nodes recursively overflows the call stack; a loop doesn't.

In [ ]:
// shared: a chain where node i's parent is i - 1
int[] chainParent(int n) {
    int[] parent = new int[n];
    for (int i = 0; i < n; i++) {
        parent[i] = i - 1;
    }
    return parent;
}

int depthRecursive(int[] parent, int node) {
    return parent[node] < 0 ? 0 : 1 + depthRecursive(parent, parent[node]);
}

In [ ]:
int[] parent = chainParent(100_000);
try {
    IO.println("recursive: " + depthRecursive(parent, 99_999));
} catch (StackOverflowError e) {
    IO.println("recursive: StackOverflowError");
}
int depth = 0;
for (int node = 99_999; parent[node] >= 0; node = parent[node]) {
    depth++;
}
IO.println("loop: " + depth);

## Readings

- [Binary Search Tree](https://visualgo.net/en/bst) (VisuAlgo (National University of Singapore), 20 min): Insert your own keys, then try Search and the in-order traversal. Insert 1, 2, 3, 4, 5 into an empty tree to see it become a chain, and note what the AVL tree in the same module does to avoid that.
- [Introduction to Tree Algorithms](https://usaco.guide/silver/intro-tree) (USACO Guide, 12 min): The vocabulary (root, parent, subtree, depth), the three traversal orders, and the Subordinates example: a node's subtree size is one plus its children's, the same idea as a folder's total size. Pick Java in the site's settings to see Java code.

## Check yourself

1. Which traversal prints a BST's keys in order?
2. What do you get by inserting 1, 2, 3, 4, 5 into an empty BST, and what is its height?
3. Why does a folder's size need a post-order traversal?

## Challenges

Solve the unit's challenges right here: each cell starts with `// sphinx: challenge …`, and running it runs **all** of the challenge's tests, hidden ones included. A pass counts in your progress in the Sphinx sidebar, just like Submit. The full description, with examples and hints, opens from the sidebar.

### Org Chart · Easy

In [ ]:
// sphinx: challenge org-chart
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] manager = new int[n + 1]; // manager[i] for employees 2..n; employee 1 is the CEO
    for (int i = 2; i <= n; i++) {
        manager[i] = Integer.parseInt(parts[i - 2]);
    }

    // TODO: compute every employee's level, then print the four lines
}

### BST Builder · Medium

In [ ]:
// sphinx: challenge bst-builder
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] keys = new int[n];
    for (int i = 0; i < n; i++) {
        keys[i] = Integer.parseInt(parts[i]);
    }

    // TODO: insert the keys into a binary search tree, then print the traversals, the height and the leaves
}

### Autocomplete · Medium

In [ ]:
// sphinx: challenge autocomplete
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] words = new String[n];
    for (int i = 0; i < n; i++) {
        words[i] = IO.readln().trim();
    }
    int q = Integer.parseInt(IO.readln().trim());
    for (int i = 0; i < q; i++) {
        String prefix = IO.readln().trim();

        // TODO: print how many words start with prefix, using a trie built from the words
    }
}

### Folder Sizes · Hard

In [ ]:
// sphinx: challenge folder-sizes
void main() {
    String[] first = IO.readln().trim().split(" +");
    int n = Integer.parseInt(first[0]);
    long limit = Long.parseLong(first[1]);
    for (int i = 0; i < n; i++) {
        String[] parts = IO.readln().trim().split(" +");
        String path = parts[0];
        long size = Long.parseLong(parts[1]);

        // TODO: add the file to a tree of folders
    }

    // TODO: add the totals up the tree (post-order) and print the five lines
}